In [1]:
import math
import os
import pprint

import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
print('TF: {}'.format(tf.__version__))

import tensorflow_data_validation as tfdv
print('TFDV version:', tfdv.version.__version__)

import apache_beam as beam
print('Beam: {}'.format(beam.__version__))

import tensorflow_transform as tft
import tensorflow_transform.beam as tft_beam
#from tensorflow_transform.keras_lib import tf_keras
print('Transform: {}'.format(tft.__version__))

from tfx_bsl.public import tfxio
from tfx_bsl.coders.example_coder import RecordBatchToExamplesEncoder

#tf.compat.v1.disable_eager_execution()

TF: 2.17.0


TFDV version: 1.14.0
Beam: 2.65.0
Transform: 1.14.0


In [2]:
df = pd.read_csv("dataset/train_cleaned.csv")
df.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight', 'Price'],
      dtype='object')

In [ ]:
# for column in df.columns:
#     print(f"========== VALUE COUNTS: [{column}] =================")
#     print(df[column].value_counts(dropna=False))
#     print(f"NULL values: {df[column].isnull().sum()}")

In [ ]:
# pd.set_option('display.max_rows', None)
# df['Weight'].value_counts().sort_index()  # alphabetical order

In [3]:
from tensorflow_metadata.proto.v0 import schema_pb2
def schema_to_feature_spec(schema):
    feature_spec = {}
    for feature in schema.feature:
        # Determine dtype
        if feature.type == schema_pb2.FeatureType.FLOAT:
            dtype = tf.float32
        elif feature.type == schema_pb2.FeatureType.INT:
            dtype = tf.int64
        elif feature.type == schema_pb2.FeatureType.BYTES:
            dtype = tf.string
        else:
            raise ValueError(f"Unsupported feature type {feature.type}")

        # Shape
        if feature.shape.dim:
            dims = [dim.size for dim in feature.shape.dim]
        else:
            dims = []

        feature_spec[feature.name] = tf.io.FixedLenFeature(
            shape=dims, dtype=dtype
        )
    return feature_spec


schema = tfdv.load_schema_text('clean_manual_schema.pbtxt')
feature_spec = schema_to_feature_spec(schema)
#RAW_DATA_FEATURE_SPEC = tfdv.utils.schema_utils.schema_as_feature_spec(schema).feature_spec

RAW_DATA_METADATA = tft.DatasetMetadata.from_feature_spec(feature_spec)

In [4]:
feature_spec

{'Price': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'Company': FixedLenFeature(shape=[], dtype=tf.string, default_value=None),
 'TypeName': FixedLenFeature(shape=[], dtype=tf.string, default_value=None),
 'Inches': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'ScreenResolution': FixedLenFeature(shape=[], dtype=tf.string, default_value=None),
 'Ram': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'Memory': FixedLenFeature(shape=[], dtype=tf.string, default_value=None),
 'OpSys': FixedLenFeature(shape=[], dtype=tf.string, default_value=None),
 'Weight': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'Cpu': FixedLenFeature(shape=[], dtype=tf.string, default_value=None),
 'Gpu': FixedLenFeature(shape=[], dtype=tf.string, default_value=None)}

In [5]:
def validate_inches(inches_input):
    inches = tf.cast(inches_input, tf.float32)
    # convert inches feature into buckets (e.g. small <14", medium <15.6", large >=15.6")
    inches_bucket = tf.where(
        inches < 14.0, 0,
        tf.where(inches < 15.6, 1, 2)
    )
    inches_bucket = tf.cast(inches_bucket, tf.float32)
    return inches_bucket / 2.0  # scale to [0,1]

In [6]:
def validate_screen_resolution(screen_resolution_input):
    screen = tf.strings.strip(tf.strings.lower(screen_resolution_input))

    screen = tf.where(
        tf.logical_or(tf.equal(screen, ''), tf.equal(screen, 'unknown')),
        tf.constant('0x0'),
        screen
    )

    # If no 'x' present, append " x 0"
    screen = tf.where(
        tf.strings.regex_full_match(screen, r'^[^x]*$'),
        tf.strings.join([screen, ' x 0']),
        screen
    )

    # Extract features
    is_ips = tf.cast(tf.strings.regex_full_match(screen, r'.*ips panel.*'), tf.int64)
    is_retina = tf.cast(tf.strings.regex_full_match(screen, r'.*retina display.*'), tf.int64)
    is_full_hd = tf.cast(tf.strings.regex_full_match(screen, r'.*full hd.*'), tf.int64)
    is_quad_hd_plus = tf.cast(tf.strings.regex_full_match(screen, r'.*quad hd\+.*'), tf.int64)
    is_touchscreen = tf.cast(tf.strings.regex_full_match(screen, r'.*touchscreen.*'), tf.int64)

    # Strip the descriptors so we have only resolution left
    screen = tf.strings.regex_replace(screen, r'ips panel', '')
    screen = tf.strings.regex_replace(screen, r'retina display', '')
    screen = tf.strings.regex_replace(screen, r'full hd', '')
    screen = tf.strings.regex_replace(screen, r'quad hd\+', '')
    screen = tf.strings.regex_replace(screen, r'4k ultra hd', '')
    screen = tf.strings.regex_replace(screen, r'touchscreen', '')
    screen = tf.strings.regex_replace(screen, r'[/]', '')  # remove slashes
    screen = tf.strings.regex_replace(screen, r'\s+', ' ')  # collapse whitespace
    screen = tf.strings.strip(screen)

    #tf.print(screen)

    width_height = tf.strings.split(screen, 'x', maxsplit=2).to_tensor(
        default_value='0', shape=[None, 2]
    )
    screen_width = tf.strings.to_number(width_height[:, 0], out_type=tf.float32)
    screen_height = tf.strings.to_number(width_height[:, 1], out_type=tf.float32)

    screen_width = tft.scale_to_z_score(screen_width)
    screen_height = tft.scale_to_z_score(screen_height)

    return {
        'is_ips': is_ips,
        'is_retina': is_retina,
        'is_full_hd': is_full_hd,
        'is_quad_hd_plus': is_quad_hd_plus,
        'is_touchscreen': is_touchscreen,
        'scaled_width': screen_width,
        'scaled_height': screen_height,
    }

In [7]:
import tensorflow as tf

@tf.function
def test_preprocessing_fn(screen_input):
    return validate_screen_resolution(screen_input)

# Sample data to test
sample_input = tf.constant([
    "Full HD 1920x1080",
    "IPS Panel 4K Ultra HD / Touchscreen 3840x2160",
    "unknown",
    "",  # blank
    "Touchscreen 1366x768",
])

result = test_preprocessing_fn(sample_input)

# Print each feature
for key, value in result.items():
    print(f"{key}: {value.numpy()}")


is_ips: [0 1 0 0 0]
is_retina: [0 0 0 0 0]
is_full_hd: [1 0 0 0 0]
is_quad_hd_plus: [0 0 0 0 0]
is_touchscreen: [0 1 0 0 1]
scaled_width: [1920. 3840.    0.    0. 1366.]
scaled_height: [1080. 2160.    0.    0.  768.]


In [8]:
def validate_cpu(cpu_inputs):
    cpu = tf.strings.lower(cpu_inputs)  # lowercase
    cpu = tf.strings.strip(cpu)

    # Base flags
    is_intel = tf.cast(tf.strings.regex_full_match(cpu, r'.*intel.*'), tf.int64)
    is_amd = tf.cast(tf.strings.regex_full_match(cpu, r'.*amd.*'), tf.int64)

    # Intel subflags
    is_i3 = tf.cast(tf.strings.regex_full_match(cpu, r'.*i3.*'), tf.int64)
    is_i5 = tf.cast(tf.strings.regex_full_match(cpu, r'.*i5.*'), tf.int64)
    is_i7 = tf.cast(tf.strings.regex_full_match(cpu, r'.*i7.*'), tf.int64)

    # AMD subflag
    is_ryzen = tf.cast(tf.strings.regex_full_match(cpu, r'.*ryzen.*'), tf.int64)

    # Fallback general processor
    is_general_processor = tf.cast(
        (is_intel + is_amd) == 0,
        tf.int64
    )

    # Extract GHz value like "2.7GHz" → "2.7"
    ghz = tf.strings.regex_replace(cpu, r'.*?([\d\.]+)\s*ghz.*', r'\1')
    ghz = tf.where(
        tf.strings.regex_full_match(ghz, r'^[\d\.]+$'),
        ghz,
        '0.0'
    )
    ghz = tf.strings.to_number(ghz, out_type=tf.float32)
    scaled_ghz = tft.scale_to_z_score(ghz)

    return {
        'is_intel_cpu': is_intel,
        'is_amd_cpu': is_amd,
        'is_general_cpu': is_general_processor,
        'is_i3_cpu': is_i3,
        'is_i5_cpu': is_i5,
        'is_i7_cpu': is_i7,
        'is_ryzen_cpu': is_ryzen,
        'scaled_ghz': scaled_ghz,
    }


In [9]:
@tf.function
def test_cpu_preprocessing_fn(cpu_input):
    return validate_cpu(cpu_input)

example_cpus = tf.constant([
    "Intel Core i5 7200U 2.7GHz",
    "Intel Core i7 7500U 2.5GHz",
    "AMD Ryzen 1600 3.2GHz",
    "unknown",
    "Samsung Cortex A72&A53 2.0GHz"
])

result = test_cpu_preprocessing_fn(example_cpus)

# Print each feature
for key, value in result.items():
    print(f"{key}: {value.numpy()}")

is_intel_cpu: [1 1 0 0 0]
is_amd_cpu: [0 0 1 0 0]
is_general_cpu: [0 0 0 1 1]
is_i3_cpu: [0 0 0 0 0]
is_i5_cpu: [1 0 0 0 0]
is_i7_cpu: [0 1 0 0 0]
is_ryzen_cpu: [0 0 1 0 0]
scaled_ghz: [2.7 2.5 3.2 0.  2. ]


In [10]:
def validate_memory(memory_inputs):
    memory = tf.strings.lower(tf.strings.strip(memory_inputs))

    # Fill unknowns with "0GB"
    memory = tf.where(
        tf.logical_or(tf.equal(memory, 'unknown'),  tf.equal(memory, '?')),
        '0GB',
        memory
    )

    # Ensure there's a '+' part
    memory = tf.where(
        tf.strings.regex_full_match(memory, r'^[^+]*$'),
        tf.strings.join([memory, ' + 0gb']),
        memory
    )

    #tf.print(memory)

    # Split into at most 2 parts
    parts = tf.strings.split(memory, sep='+', maxsplit=1).to_tensor(default_value='0gb')
    part0 = parts[:, 0]
    part1 = parts[:, 1]

    def part_to_gb(part):
        num = tf.strings.regex_replace(part, r'([0-9\.]+).*', r'\1')
        num = tf.strings.to_number(num, out_type=tf.float32)

        # Check if part contains TB
        is_tb = tf.strings.regex_full_match(part, r'.*[Tt][Bb].*')
        is_tb = tf.cast(is_tb, tf.float32)  # ✅ CAST to float32

        # Multiply num by 1024 if it's TB, or 1 if it's GB
        return num * (is_tb * 1024.0 + (1.0 - is_tb))

    num_gb = part_to_gb(part0) + part_to_gb(part1)

    # Fill unknown with a default, e.g. median SSD size = 256GB
    median_ssd = 256.0
    num_gb = tf.where(
        num_gb == 0.0,
        tf.constant(median_ssd, dtype=tf.float32),
        num_gb
    )

    return {
        'scaled_total_memory': num_gb,
        'is_ssd': tf.cast(tf.strings.regex_full_match(memory, r'.*ssd.*'), tf.int64),
        'is_hdd': tf.cast(tf.strings.regex_full_match(memory, r'.*hdd.*'), tf.int64),
        'is_flash': tf.cast(tf.strings.regex_full_match(memory, r'.*flash.*'), tf.int64),
        'is_hybrid': tf.cast(tf.strings.regex_full_match(memory, r'.*hybrid.*'), tf.int64),
    }

In [11]:
@tf.function
def test_memory_preprocessing_fn(mem_input):
    return validate_memory(mem_input)

example_mem = tf.constant([
    "256GB SSD",
    "1TB HDD",
    "128GB SSD + 1TB HDD",
    "unknown",
    "?",
    "512GB Flash Storage + 2TB HDD",
    "64GB Flash Storage",
    "256GB SSD + 256GB SSD",
])

result = test_memory_preprocessing_fn(example_mem)

# Print each feature
for key, value in result.items():
    print(f"{key}: {value.numpy()}")

scaled_total_memory: [ 256. 1024. 1152.  256.  256. 2560.   64.  512.]
is_ssd: [1 0 1 0 0 0 0 1]
is_hdd: [0 1 1 0 0 1 0 0]
is_flash: [0 0 0 0 0 1 1 0]
is_hybrid: [0 0 0 0 0 0 0 0]


In [12]:
def validate_gpu(gpu_input):
    gpu_str = tf.strings.lower(tf.strings.strip(gpu_input))

    gpu_str = tf.where(
        tf.strings.regex_full_match(gpu_str, r'.*invalid.*'),
        'unknown',
        gpu_str
    )

    # Gpu subflags
    is_amd_radeon_gpu = tf.cast(tf.strings.regex_full_match(gpu_str, r'.*amd radeon.*'), tf.int64)
    is_intel_hd_gpu = tf.cast(tf.strings.regex_full_match(gpu_str, r'.*intel hd.*'), tf.int64)
    is_intel_iris_gpu = tf.cast(tf.strings.regex_full_match(gpu_str, r'.*intel iris.*'), tf.int64)
    is_nvidia_geforce_gpu = tf.cast(tf.strings.regex_full_match(gpu_str, r'.*nvidia geforce.*'), tf.int64)
    is_nvidia_quadro_gpu = tf.cast(tf.strings.regex_full_match(gpu_str, r'.*nvidia quadro.*'), tf.int64)

    # Fallback general processor
    is_other_gpu = tf.cast(
        (is_amd_radeon_gpu + is_intel_hd_gpu + is_intel_iris_gpu + is_nvidia_geforce_gpu + is_nvidia_quadro_gpu) == 0,
        tf.int64
    )

    return {
        "is_amd_radeon_gpu": is_amd_radeon_gpu,
        "is_intel_hd_gpu": is_intel_hd_gpu,
        "is_intel_iris_gpu": is_intel_iris_gpu,
        "is_nvidia_geforce_gpu": is_nvidia_geforce_gpu,
        "is_nvidia_quadro_gpu": is_nvidia_quadro_gpu,
        "is_other_gpu": is_other_gpu
    }


In [13]:
@tf.function
def test_gpu_preprocessing_fn(gpu):
    return validate_gpu(gpu)

example_gpu = tf.constant([
    "###invalid###",
    "AMD FirePro W4190M",
    "AMD R4 Graphics",
    "unknown",
    "AMD Radeon Pro 555",
    "Intel HD Graphics 520",
    "Intel Iris Graphics 550",
    "Nvidia GeForce 920MX ",
    "Nvidia Quadro M1200"
])

result = test_gpu_preprocessing_fn(example_gpu)

# Print each feature
for key, value in result.items():
    print(f"{key}: {value.numpy()}")

is_amd_radeon_gpu: [0 0 0 0 1 0 0 0 0]
is_intel_hd_gpu: [0 0 0 0 0 1 0 0 0]
is_intel_iris_gpu: [0 0 0 0 0 0 1 0 0]
is_nvidia_geforce_gpu: [0 0 0 0 0 0 0 1 0]
is_nvidia_quadro_gpu: [0 0 0 0 0 0 0 0 1]
is_other_gpu: [1 1 1 1 0 0 0 0 0]


In [14]:
def preprocessing_fn(inputs):
    outputs = {}

    # --- 1. Company (categorical) ---
    outputs['company_xf'] = tft.compute_and_apply_vocabulary(inputs['Company'])

    # --- 2. TypeName (categorical) ---
    type_name = tf.strings.lower(inputs["TypeName"])
    type_name = tf.strings.strip(type_name)
    outputs['typename_xf'] = tft.compute_and_apply_vocabulary(type_name)

    #--- 3. Inches (bucketing) ---
    inches =validate_inches(inputs["Inches"])
    outputs['scaled_inches'] = tf.cast(inches, tf.float32)

    # # --- 4. ScreenResolution (numerical) --
    screen = validate_screen_resolution(inputs["ScreenResolution"])
    for key, value in screen.items():
        outputs[key] = value


    #--- 5. CPU (categorical) --
    # Process CPU
    cpu = validate_cpu(inputs["Cpu"])
    for key, value in cpu.items():
        outputs[key] = value

    #--- 6. RAM (numerical) --
    outputs["scaled_ram"] = tft.scale_to_z_score(inputs["Ram"])

    #---7. Memory () ---
    memory = validate_memory(inputs["Memory"])
    for key, value in memory.items():
        outputs[key] = value


    #--- 8. GPU (categorical) ---
    gpu = validate_gpu(inputs['Gpu'])
    for key, value in gpu.items():
        outputs[key] = value

    # --- 9. OpSys (categorical) ---
    def normalize_os_single(os_str):
        os_str = tf.strings.lower(tf.strings.strip(os_str))
        is_windows = tf.strings.regex_full_match(os_str, '.*windows.*')
        is_mac = tf.strings.regex_full_match(os_str, '.*mac.*')

        return tf.case([
            (is_windows, lambda: tf.constant("windows")),
            (is_mac, lambda: tf.constant("macos")),
        ], default=lambda: os_str)

    normalized_opsys = tf.map_fn(
        normalize_os_single,
        inputs["OpSys"],
        fn_output_signature=tf.TensorSpec([], tf.string)
    )
    outputs['opsys_xf'] = tft.compute_and_apply_vocabulary(normalized_opsys)

    #---10. Weight (numerical) ---
    outputs['scaled_weight'] = tft.scale_to_z_score(inputs['Weight'])

    #---11. Price (numerical) ---
    outputs["scaled_price"] = tft.scale_to_z_score(inputs["Price"])


    return outputs



In [15]:
import apache_beam as beam
import tensorflow_transform.beam as tft_beam
import tempfile
import csv
import numpy as np
import random

class Split(beam.DoFn):
    def process(self, element):
        try:
            parts = element.strip().split(",")
            if len(parts) != 11:
                return  # skip malformed rows

            Company, TypeName, Inches, ScreenResolution, Cpu, Ram, Memory, Gpu, OpSys, Weight, Price = parts

            def safe_float(val):
                try:
                    return float(val.strip())
                except:
                    return np.nan  # use NaN for invalid values like "unknown"

            yield {
                "Company": Company.strip(),
                "TypeName": TypeName.strip(),
                "Inches": safe_float(Inches),
                "ScreenResolution": ScreenResolution.strip(),
                "Cpu": Cpu.strip(),
                "Ram": safe_float(Ram),
                "Memory": Memory.strip(),
                "Gpu": Gpu.strip(),
                "OpSys": OpSys.strip(),
                "Weight": safe_float(Weight),
                "Price": safe_float(Price),  # <- safely handle unknown/invalid
            }

        except Exception as e:
            return  # optionally log or skip

In [17]:
def run_pipeline(input_csv, output_prefix, transform_fn_dir=None, analyze=False):
    with beam.Pipeline() as pipeline:
        with tft_beam.Context(temp_dir="./tmp"):
            raw_data = (
                pipeline
                | f"Read CSV {input_csv}" >> beam.io.ReadFromText(input_csv, skip_header_lines=1)
                | f"Parse CSV {input_csv}" >> beam.ParDo(Split())
            )

            dataset = (raw_data, RAW_DATA_METADATA)

            if analyze:
                # Analyze & Transform for training data
                transformed_dataset, transform_fn = (
                    dataset | "Analyze and Transform Train Data" >> tft_beam.AnalyzeAndTransformDataset(preprocessing_fn)
                )
                transformed_data, transformed_metadata = transformed_dataset

                # Save transform_fn
                _ = (
                    transform_fn
                    | "Write TransformFn" >> tft_beam.WriteTransformFn(os.path.join(output_prefix))
                )

            else:
                transform_fn = pipeline | "Read TransformFn" >> tft_beam.ReadTransformFn(transform_fn_dir)

                transformed_data, transformed_metadata = (
                    ((raw_data, RAW_DATA_METADATA), transform_fn)
                    | f"Transform {input_csv}" >> tft_beam.TransformDataset()
                )

            # (Optional) Print to check
            #_ = transformed_data | f"Print {output_prefix}" >> beam.Map(lambda x: print(f"{output_prefix}:", x) or x)

            _ = (
                transformed_data
                | "Reshuffle Before Write" >> beam.transforms.util.Reshuffle()
                | f"Write TFRecords {output_prefix}" >> beam.io.WriteToTFRecord(
                    file_path_prefix=output_prefix,
                    file_name_suffix='.gz',
                    coder=tft.coders.example_proto_coder.ExampleProtoCoder(transformed_metadata.schema)
                )
            )


In [18]:
# --- Run for Train (Analyze + Transform) ---
run_pipeline(
    input_csv="dataset/train_cleaned.csv",
    output_prefix="train",
    analyze=True
)

INFO:tensorflow:Assets written to: ./tmp\tftransform_tmp\bc145537d7dd4b4e8cf1237debb0639d\assets


INFO:tensorflow:Assets written to: ./tmp\tftransform_tmp\bc145537d7dd4b4e8cf1237debb0639d\assets


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


In [19]:
run_pipeline(
    input_csv="dataset/eval_cleaned.csv",
    output_prefix="eval",
    transform_fn_dir="train",
    analyze=False
)

INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


In [21]:
run_pipeline(
    input_csv="dataset/test_cleaned.csv",
    output_prefix="test",
    transform_fn_dir="train",
    analyze=False
)

INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.
